In [1]:
import pandas as pd
import numpy as np
import torch
import time
import copy
import math
import os
import psutil 
from datetime import datetime
from torch.utils.data import DataLoader

In [3]:
data = pd.read_csv('/Users/nkerstingadxnet.com/Documents/Higgs/orig/atlas-higgs-challenge-2014-v2.csv')
data['Binary_Label'] = data['Label'].map({'s':1,'b':0})

nonphi_columns = [c for c in data.columns if "phi" in c]
for col in nonphi_columns:
    data.drop(col, axis=1, inplace=True)
data.head()

# now replace all the '-999' values with NULL so we can properly handle them
#nulled_data = data.replace(-999.0, np.nan)
nulled_data = data.replace(-999.0, 0)
train_data = nulled_data.loc[nulled_data['KaggleSet'] == 't'].copy(deep=True)
public_test_data = nulled_data.loc[nulled_data['KaggleSet'] == 'b'].copy(deep=True)
private_test_data = nulled_data.loc[nulled_data['KaggleSet'] == 'v'].copy(deep=True)

orig_train_data = train_data.copy(deep=True)
orig_public_test_data = public_test_data.copy(deep=True)
orig_private_test_data = private_test_data.copy(deep=True)


# now replace the null values with column averages
train_data.drop('EventId', axis=1, inplace=True)
train_data.drop('Label', axis=1, inplace=True)
train_data.drop('Binary_Label', axis=1, inplace=True)
train_data.drop('Weight', axis=1, inplace=True)
train_data.drop('KaggleWeight', axis=1, inplace=True)
train_data.drop('KaggleSet', axis=1, inplace=True)
for col in train_data:
    train_data[col].fillna(train_data[col].mean(), inplace=True)
#train_data.isnull().sum()


public_test_data.drop('EventId', axis=1, inplace=True)
public_test_data.drop('Label', axis=1, inplace=True)
public_test_data.drop('Binary_Label', axis=1, inplace=True)
public_test_data.drop('Weight', axis=1, inplace=True)
public_test_data.drop('KaggleWeight', axis=1, inplace=True)
public_test_data.drop('KaggleSet', axis=1, inplace=True)
for col in public_test_data:
    public_test_data[col].fillna(public_test_data[col].mean(), inplace=True)
#public_test_data.isnull().sum()


private_test_data.drop('EventId', axis=1, inplace=True)
private_test_data.drop('Label', axis=1, inplace=True)
private_test_data.drop('Binary_Label', axis=1, inplace=True)
private_test_data.drop('Weight', axis=1, inplace=True)
private_test_data.drop('KaggleWeight', axis=1, inplace=True)
private_test_data.drop('KaggleSet', axis=1, inplace=True)
for col in private_test_data:
    private_test_data[col].fillna(private_test_data[col].mean(), inplace=True)
#private_test_data.isnull().sum()

# now let's normalize
normed_train_data = (train_data - train_data.min())/(train_data.max() - train_data.min())

normed_public_test_data = (public_test_data - public_test_data.min())/(public_test_data.max() - public_test_data.min())

normed_private_test_data = (private_test_data - private_test_data.min())/(private_test_data.max() - private_test_data.min())


train_input_data = []
for i in range(len(orig_train_data)):
   train_input_data.append([torch.tensor(normed_train_data.iloc[i], dtype=torch.float), torch.tensor(orig_train_data['Binary_Label'].iloc[i] , dtype=torch.float)])
valid_input_data = []
for i in range(len(orig_public_test_data)):
   valid_input_data.append([torch.tensor(normed_public_test_data.iloc[i], dtype=torch.float), torch.tensor(orig_public_test_data['Binary_Label'].iloc[i] , dtype=torch.float)])

In [4]:
class Feedforward(torch.nn.Module):
        def __init__(self, input_size, hidden_size, dropout):
            super(Feedforward, self).__init__()
            self.input_size = input_size
            self.hidden_size  = hidden_size
            self.fc_in = torch.nn.Linear(self.input_size, self.hidden_size)
            self.fc1 = torch.nn.Linear(self.hidden_size, self.hidden_size)
            self.fc2 = torch.nn.Linear(self.hidden_size, self.hidden_size)
            #self.relu = torch.nn.ReLU()
            self.relu = torch.nn.LeakyReLU(0.1)
            self.fc_out = torch.nn.Linear(self.hidden_size, 1)
            self.sigmoid = torch.nn.Sigmoid()
            self.dropout = torch.nn.Dropout(dropout)
            
        def forward(self, x):
            x = self.fc_in(x)
            x = self.dropout(x)
            x = self.relu(x)
            x = self.fc1(x)
            x = self.dropout(x)
            x = self.relu(x)
            x = self.fc2(x)
            x = self.dropout(x)
            x = self.relu(x)
            output = self.fc_out(x)
            output = self.sigmoid(output)
            return output

In [5]:
train_dataloader = DataLoader(train_input_data, batch_size=128, shuffle=True)
valid_dataloader = DataLoader(valid_input_data, batch_size=128, shuffle=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = Feedforward(24, 600, 0.05)
model.to(device)
criterion = torch.nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(), lr = 0.01)

In [6]:
normed_private_test_data.head()

,DER_mass_MMC,DER_mass_transverse_met_lep,DER_mass_vis,DER_pt_h,DER_deltaeta_jet_jet,DER_mass_jet_jet,DER_prodeta_jet_jet,DER_deltar_tau_lep,DER_pt_tot,DER_sum_pt,...,PRI_lep_pt,PRI_lep_eta,PRI_met,PRI_met_sumet,PRI_jet_num,PRI_jet_leading_pt,PRI_jet_leading_eta,PRI_jet_subleading_pt,PRI_jet_subleading_eta,PRI_jet_all_pt
250001,0.054584,0.069673,0.064490,0.046307,0.000000,0.000000,0.530928,0.328437,0.006267,0.042676,...,0.051522,0.466494,0.028886,0.074620,0.333333,0.045682,0.438653,0.000000,0.500000,0.025576
250002,0.060430,0.058045,0.071174,0.003832,0.000000,0.000000,0.530928,0.456656,0.009678,0.025330,...,0.045143,0.390905,0.026646,0.044871,0.000000,0.000000,0.500111,0.000000,0.500000,0.000000
250003,0.069699,0.031594,0.071913,0.008433,0.000000,0.000000,0.530928,0.466812,0.021298,0.023614,...,0.029519,0.441364,0.034430,0.069315,0.000000,0.000000,0.500111,0.000000,0.500000,0.000000
250004,0.038045,0.085449,0.041268,0.083035,0.154402,0.111925,0.521461,0.143453,0.180631,0.332233,...,0.092413,0.118668,0.030406,0.393198,1.000000,0.243977,0.387531,0.227264,0.537222,0.322127
250005,0.049100,0.097214,0.047268,0.013634,0.000000,0.000000,0.530928,0.512876,0.034433,0.011882,...,0.043244,0.374551,0.061136,0.033211,0.000000,0.000000,0.500111,0.000000,0.500000,0.000000


In [7]:
now = datetime.now()
dt_string = now.strftime("%d%m%Y%H%M%S")
logfile = open(dt_string + ".log",'w')
logfile.write(f"Start of Log\n")
logfile.write(f"Now training with device={device}\n")
#logfile.write(f"Num Devices: {torch.cuda.device_count()}, DEVICE NAME 0: {torch.cuda.get_device_name(0)}\n")

start_time = time.time()
epochs = 1
maxvalcount = 5
minval_loss = np.inf
valcount = maxvalcount
best_model = model
logfile.write(f"Using model with params on CUDA: {next(model.parameters()).is_cuda}\n")
for epoch in range(epochs):
    outstring = f"EPOCH: {epoch}"
    print(outstring)
    logfile.write(outstring + '\n')
    model.train()
    for i,batch in enumerate(train_dataloader):
        inputs, output = batch
        inputs, output = inputs.to(device), output.to(device)
        optimizer.zero_grad()
        # Forward pass
        y_pred = model(inputs)
        # Compute Loss
        loss = criterion(y_pred.squeeze(), output)
        if i % 100 == 0:
            logstring = 'Batch {}: train loss: {}'.format(i, loss.item())
            print(logstring)
            logfile.write(logstring + '\n')
        # Backward pass
        loss.backward()
        optimizer.step()
    # compute validation loss
    model.eval()
    with torch.set_grad_enabled(False):
        val_loss = 0
        for i,batch in enumerate(valid_dataloader):
            inputs, output = batch
            inputs, output = inputs.to(device), output.to(device)
            y_pred = model(inputs)
            loss = criterion(y_pred.squeeze(), output)
            val_loss += loss
            if i % 100 == 0:
                logstring = 'Validation Batch {}: loss: {}'.format(i, loss.item())
                print(logstring)
                logfile.write(logstring + '\n')
        avg_val_loss = val_loss / len(valid_dataloader)
        valstring = f"AVERAGE BATCH VAL LOSS = {avg_val_loss}"
        print(valstring)
        logfile.write(valstring + '\n')
    if avg_val_loss < minval_loss:
        minval_loss = avg_val_loss
        best_model = copy.deepcopy(model)
        valcount = maxvalcount
    else:
        valcount -= 1
    if valcount == 0:
        endmsg = f"Validation Loss failed to decrease in {maxvalcount} epochs, exiting with best model, val loss = {minval_loss}"
        print(endmsg)
        logfile.write(endmsg + '\n')
        break

end_time = time.time()
timestring = f"Time elapsed in training: {end_time - start_time} seconds"
print(timestring)
logfile.write(timestring + '\n')

EPOCH: 0
Batch 0: train loss: 0.696816623210907
Batch 100: train loss: 0.6500580310821533
Batch 200: train loss: 0.6790027022361755
Batch 300: train loss: 0.6565102934837341
Batch 400: train loss: 0.6069412231445312
Batch 500: train loss: 0.6084862947463989
Batch 600: train loss: 0.6472079157829285
Batch 700: train loss: 0.6233185529708862
Batch 800: train loss: 0.6151746511459351
Batch 900: train loss: 0.6151500940322876
Batch 1000: train loss: 0.6489453315734863
Batch 1100: train loss: 0.5894832015037537
Batch 1200: train loss: 0.6119170188903809
Batch 1300: train loss: 0.6617540121078491
Batch 1400: train loss: 0.6292824745178223
Batch 1500: train loss: 0.6348507404327393
Batch 1600: train loss: 0.6097414493560791
Batch 1700: train loss: 0.5948259830474854
Batch 1800: train loss: 0.6456041932106018
Batch 1900: train loss: 0.6150934100151062
Validation Batch 0: loss: 0.6096493601799011
Validation Batch 100: loss: 0.6497904062271118
Validation Batch 200: loss: 0.6242625117301941
Valid

52

In [8]:
torch.save(best_model, 'mlp.shallow.600.nophi.pt')

In [9]:
def AMS(s, b):
    """ Approximate Median Significance defined as:
        AMS = sqrt(
                2 { (s + b + b_r) log[1 + (s/(b+b_r))] - s}
              )        
    where b_r = 10, b = background, s = signal, log is natural logarithm """
    
    br = 10.0
    radicand = 2 *( (s+b+br) * math.log (1.0 + s/(b+br)) -s)
    if radicand < 0:
        print('radicand is negative. Exiting')
        exit()
    else:
        return math.sqrt(radicand)

In [10]:
def compute_performance(logfile, setname, y_pred, y_gold, y_weight):
    s = 0
    b = 0
    tp = 0
    fp = 0
    for i,y in enumerate(y_pred):
        if y[0] > 0.5:
            if y_gold.iloc[i] == 's':
                s += y_weight.iloc[i]
                tp += 1
            else:
                b += y_weight.iloc[i]
                fp += 1

    out1 = f"{setname}: (S,B) = ({s:.3f},{b:.3f}), (TP,FP) = {tp}, {fp}"
    print(out1)
    logfile.write(out1 + '\n')

    outstring = f"{setname}: (S,B) = ({s:.3f},{b:.3f}), AMS={AMS(s,b):.3f}, Unweighted Precision = {tp/(tp+fp):.3f}"
    print(outstring)
    logfile.write(outstring + '\n')

In [11]:
y_pred = best_model(torch.tensor(normed_train_data.values, dtype=torch.float).to(device)).tolist()
y_gold = orig_train_data['Label']
y_weight = orig_train_data['KaggleWeight']
compute_performance(logfile, "TRAINING", y_pred, y_gold, y_weight)

TRAINING: (S,B) = (24.006,1309.392), (TP,FP) = 10630, 2860
TRAINING: (S,B) = (24.006,1309.392), AMS=0.659, Unweighted Precision = 0.788


In [12]:
y_pred = best_model(torch.tensor(normed_public_test_data.values, dtype=torch.float).to(device)).tolist()
y_gold = orig_public_test_data['Label']
y_weight = orig_public_test_data['KaggleWeight']
compute_performance(logfile, "VALIDATION", y_pred, y_gold, y_weight)

VALIDATION: (S,B) = (24.453,1612.553), (TP,FP) = 4448, 1288
VALIDATION: (S,B) = (24.453,1612.553), AMS=0.606, Unweighted Precision = 0.775


In [13]:
# Test set performance
y_pred = best_model(torch.tensor(normed_private_test_data.values, dtype=torch.float).to(device)).tolist()
y_gold = orig_private_test_data['Label']
y_weight = orig_private_test_data['KaggleWeight']
compute_performance(logfile, "TESTING", y_pred, y_gold, y_weight)

TESTING: (S,B) = (22.552,1270.870), (TP,FP) = 18153, 4905
TESTING: (S,B) = (22.552,1270.870), AMS=0.628, Unweighted Precision = 0.787


In [14]:
logfile.close()